# Human validation supplementary figures S4–S5

Run the setup and table-loading cells before the desired panels. Panel dimensions are in millimeters. The notebook exports figures and values used in the supplementary tables.

In [ ]:
from pathlib import Path
import sys

start = Path.cwd().resolve()
root = next(path for path in (start, *start.parents) if (path / "supplementary/supp_plotting").is_dir())
sys.path.insert(0, str(root))

import pandas as pd

from supplementary.supp_plotting.editing import (
    export_combined_function_panels, export_human_panel, export_panel, figure_size,
)
from supplementary.supp_plotting.human import (
    FUNCTION_TERMS, MAIN_THEMES, build_top50_annotations, longitudinal_display_table,
    metastatic_risk_table, plot_cd8score_functional_matrix, plot_longitudinal_patient_trajectories,
    plot_metastatic_km, plot_top50_function_annotation, select_enrichment_terms,
)
from supplementary.supp_plotting.human_combined import plot_combined_function_panels
from supplementary.supp_plotting.paths import HUMAN_RESULTS, figure_output_dir

OUTPUT_DIR = figure_output_dir("human_validation")

## Load results and prepare plot data

In [ ]:
tcga = HUMAN_RESULTS / "TCGA_SKCM"
gene = tcga / "gene_level_cd8score"
enrichment = gene / "all_cd8score_enrichment"
longitudinal_dir = HUMAN_RESULTS / "GSE272993_longitudinal/results/cd8score_longitudinal_validation"

km_curves = pd.read_csv(tcga / "outputs/km_curve_source_data.csv")
km_stats = pd.read_csv(tcga / "outputs/km_median_split_results.csv")
patients = pd.read_csv(tcga / "outputs/tcga_skcm_patient_scores.csv")
longitudinal = pd.read_csv(longitudinal_dir / "human_patient_delta_cd8score.csv")
longitudinal_summary = pd.read_csv(longitudinal_dir / "human_delta_cd8score_summary.csv")
longitudinal_display = longitudinal_display_table(longitudinal, longitudinal_summary)

all_enrichment = pd.read_csv(enrichment / "all_cd8score_enrichment_results.csv")
enrichment_display = select_enrichment_terms({
    "All CD8score": all_enrichment,
    "Positive weights": pd.read_csv(enrichment / "positive_weight_enrichment_results.csv"),
    "Negative weights": pd.read_csv(enrichment / "negative_weight_enrichment_results.csv"),
}, FUNCTION_TERMS)
top50 = pd.read_csv(gene / "top_gene_enrichment/top50_gene_sets.csv")
top50_annotations = build_top50_annotations(top50, all_enrichment, MAIN_THEMES)

## S4c — Survival in the metastatic TCGA subset

The curves and log-rank P value come from saved results. Counts at risk use the original patient follow-up data and group threshold.

In [ ]:
risk_counts = metastatic_risk_table(patients, km_stats)
fig = plot_metastatic_km(km_curves, km_stats, risk_counts, figsize=figure_size(105, 75), font_size=10)
export_panel(fig, "Human_Score_A_TCGA_metastatic_KM", output_dir=OUTPUT_DIR)

## S4d — Individual changes and mean ± SEM for four patients

The plot shows anti-PD-1-weighted ΔSW-CD8 for four patients across five visits. Light lines show individuals, and the black line with error bars shows the saved mean ± SEM.

In [ ]:
fig = plot_longitudinal_patient_trajectories(
    longitudinal, longitudinal_summary, figsize=figure_size(105, 75), font_size=10,
)
export_human_panel(fig, "Human_Score_E_GSE272993_patient_trajectories", longitudinal_display, output_dir=OUTPUT_DIR)

## S5a — Functional composition of all, positively weighted and negatively weighted genes

The three columns share 13 themes. Color encodes BH-FDR across the full library, and bubble area shows the proportion within each gene set. Open circles indicate q ≥ 0.05, and dashes indicate no overlap.

In [ ]:
fig = plot_cd8score_functional_matrix(enrichment_display, figsize=figure_size(155, aspect_ratio=1.10), font_size=8)
export_human_panel(fig, "Human_Function_A_CD8score_enrichment_matrix", enrichment_display, output_dir=OUTPUT_DIR)

## S5b — Top 50 genes and functional themes from the main figure

The plot shows all 50 genes in their original rank order by absolute β. The left axis shows signed β, and the right axis shows membership in eight functional themes.

In [ ]:
fig = plot_top50_function_annotation(top50_annotations, figsize=figure_size(183, aspect_ratio=0.80), font_size=7.5)
export_human_panel(fig, "Human_Function_B_Top50_function_annotation", top50_annotations, output_dir=OUTPUT_DIR)

## Combined S5 functional panels (single-page PDF, 21 cm wide)

The upper panel shows all enrichment results for 13 themes across three gene sets.
The lower panel places all 50 genes on the horizontal axis and aligns their original coefficients with all 400 membership entries for eight themes.
Both panels also export as separate figures.

In [ ]:
COMBINED_SIZE = dict(width_mm=210, height_mm=200)
COMBINED_STYLE = dict(
    font_size=9, gene_font_size=8.5, panel_label_size=14,
    color_max=32, bubble_scale=800, term_wrap=44,
    beta_ylim=(-0.025, 0.040), beta_height_scale=0.60,
)

combined_fig = plot_combined_function_panels(
    enrichment_display, top50_annotations, **COMBINED_SIZE, **COMBINED_STYLE,
)
export_combined_function_panels(combined_fig, OUTPUT_DIR)